## Dynamic Programming

#### Download necessary modules

In [166]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#### Predicting hourly demand based on 2018 data

In [167]:
# Load cleaned data (2018)
rideData2018 = pd.read_csv("../data/cleanData/df2_2018(clean_parks_metadate).csv")
rideData2018.head()

,date,wdw_ticket_season,dayofweek,dayofyear,weekofyear,monthofyear,year,season,holiday,wdwticketseason,...,hsfirewks,akprdday,akprddt1,akprddt2,akprddn,akfiren,akshwngt,akshwnt1,akshwnt2,akshwnn
0,2018-01-01,peak,2,0,0,1,2018,CHRISTMAS PEAK,1,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
1,2018-01-02,peak,3,1,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
2,2018-01-03,peak,4,2,0,1,2018,CHRISTMAS,0,peak,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
3,2018-01-04,regular,5,3,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light
4,2018-01-05,regular,6,4,0,1,2018,CHRISTMAS,0,regular,...,1,0,NaN,NaN,NaN,NaN,2,66600.0,71100.0,Rivers of Light


In [168]:
# Load waiting times data
waitTimes = pd.read_csv("../data/cleanData/animal_kingdom_df1(touringplans_2018).csv")
waitTimes.head()

,park_date,wait_hour,attraction_name,wait_minutes_posted_avg,attraction_duration,attraction_park,attraction_land,park_open,park_close,park_extra_magic_morning,park_extra_magic_evening,park_ticket_season,park_temperature_average,park_temperature_high,attraction_short_name
0,2018-01-01,8,DINOSAUR,15.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
1,2018-01-01,9,DINOSAUR,18.333333,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
2,2018-01-01,10,DINOSAUR,23.750000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
3,2018-01-01,11,DINOSAUR,24.000000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR
4,2018-01-01,12,DINOSAUR,31.875000,3.5,Disney's Animal Kingdom,DinoLand U.S.A.,8:00:00,20:00:00,0,0,peak,50.56,58.63,DINOSAUR


In [169]:
# Fun Factor
# Fun factor "a" calculation
a_dict = {'Kanika': {'Speed': 6, 'Music': 7, 'Show': 2, 'Light': 5, 'Water': 8, '3D': 10},
          'Lydia': {'Speed': 6, 'Music': 3, 'Show': 1, 'Light': 7, 'Water': 1, '3D': 3},
          'Shawn': {'Speed': 10, 'Music': 0, 'Show': 0, 'Light': 0, 'Water': 3, '3D': 2},
          'Clarice': {'Speed': 4, 'Music': 5, 'Show': 3, 'Light': 2, 'Water': 3, '3D': 5},
          'Ethan': {'Speed': 5, 'Music': 6, 'Show': 2, 'Light': 1, 'Water': 4, '3D': 6},
          'Kevin': {'Speed': 6, 'Music': 7, 'Show': 1, 'Light': 0, 'Water': 5, '3D': 7},
          'Allison': {'Speed': 7, 'Music': 8, 'Show': 0, 'Light': 1, 'Water': 6, '3D': 8}}

# Load attributes of rides 
rideAttributes = pd.read_excel("../data/cleanData/Book1.xlsx")

# Func factor calculation
def calculate_fun_factor(person):
    ride_fun_factors = {}
    for ride in rideAttributes['Ride']:
        fun_factor = 0
        attributes = rideAttributes[rideAttributes['Ride'] == ride].iloc[0]
        fun_factor += (a_dict[person]['Speed'] * attributes['Speed'] +
                       a_dict[person]['Music'] * attributes['Music'] +
                       a_dict[person]['Show'] * attributes['Show'] +
                       a_dict[person]['Light'] * attributes['Light'] +
                       a_dict[person]['Water'] * attributes['Water'] +
                       a_dict[person]['3D'] * attributes['3D'])
        ride_fun_factors[ride] = float(fun_factor)
    return ride_fun_factors

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
calculate_fun_factor('Kanika')

{'DINOSAUR': 54.0,
 'Expedition Everest - Legend of the Forbidden Mountain': 57.0,
 'Avatar Flight of Passage': 100.0,
 'Kilimanjaro Safaris': 50.0,
 "Na'vi River Journey": 82.0}

In [ ]:
# Create average wait time per attraction
average_wait_times = waitTimes.groupby(['wait_hour', 'attraction_name'])['wait_minutes_posted_avg'].mean().reset_index()
average_wait_times.columns = ['wait_hour', 'attraction_name', 'average_wait_time']
average_wait_times.head()

,wait_hour,attraction_name,average_wait_time
0,4,DINOSAUR,5.000000
1,4,Kilimanjaro Safaris,31.000000
2,5,Kilimanjaro Safaris,51.000000
3,6,Avatar Flight of Passage,65.803571
4,6,DINOSAUR,5.000000


In [ ]:
# Attraction Duration for each ride
attraction_durations = waitTimes[['attraction_name', 'attraction_duration']].rename(columns={'attraction_name': 'Ride', 'attraction_duration': 'Duration'})
# Only keep unique rides
attraction_durations = attraction_durations.drop_duplicates(subset=['Ride'])
# Change Index to Ride
attraction_durations = attraction_durations.set_index('Ride')
attraction_durations.head()

,Duration
Ride,
DINOSAUR,3.5
Expedition Everest - Legend of the Forbidden Mountain,4.0
Avatar Flight of Passage,6.0
Kilimanjaro Safaris,20.0
Na'vi River Journey,5.0


In [ ]:
rideAttributes['Ride'].tolist()

['DINOSAUR',
 'Expedition Everest - Legend of the Forbidden Mountain',
 'Avatar Flight of Passage',
 'Kilimanjaro Safaris',
 "Na'vi River Journey"]

In [ ]:
# Building DP Model
def build_dp(
    T=20*60,      # number of hours (stages)
    alpha = 0.8, # cost per unit of wait time
    beta = 0.4,    # cost per unit of ride time
    person ='Lydia', # person for fun factor
    duration = attraction_durations,
    average_wait_times = average_wait_times,
):
    rides = attraction_durations.index.tolist()
    ride_durations = duration['Duration'].tolist()
    fun_factors = calculate_fun_factor(person)
    
    # Initialize DP table
    dp = np.zeros((int(T/60) + 1*60, len(rides)))
    path = np.zeros((int(T/60) + 1*60, len(rides)), dtype=int)
    
    for t in range(T+(1*60), 8*60, -60):
        for i, ride in enumerate(rides):
            ride_duration = ride_durations[i]  # convert to hours
            if t >= ride_duration:
                wait_time = average_wait_times[(average_wait_times['attraction_name'] == ride) & (average_wait_times['wait_hour'] == int(t - (1*60)))]['average_wait_time'].values
                wait_time = wait_time[0] if len(wait_time) > 0 else 0  # convert to hours
                cost = alpha * wait_time + beta * ride_duration
                value = fun_factors[ride] - cost
                print(dp[int(t / 60)][i])
                                
                if dp[int(t / 60)][i] < dp[int((t - int(ride_duration)) / 60)][i] + value:
                    dp[int(t / 60)][i] = dp[int((t - int(ride_duration)) / 60)][i] + value
                    path[int(t / 60)][i] = i
    
    # Backtrack to find optimal path
    optimal_path = []
    t = T
    while int(t / 60) > 8:
        i = np.argmax(dp[int(t / 60)])
        optimal_path.append(rides[i])
        # t -= int(wait_time[i])
        wt = average_wait_times[(average_wait_times['attraction_name'] == rides[i]) & (average_wait_times['wait_hour'] == int(t / 60))]['average_wait_time'].values[0]
        t = t - int(wt)
        t = t - int(ride_durations[i])
        # print(dp[int(t / 60)])
            
    optimal_path.reverse()
    return optimal_path, dp[int(T / 60)].max()

In [ ]:
build_dp()

0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck
0.0
f*ck


(['Avatar Flight of Passage',
  'Avatar Flight of Passage',
  'Avatar Flight of Passage',
  'Avatar Flight of Passage',
  'Avatar Flight of Passage'],
 np.float64(53.6))